# 01 量化交易全流程与主线案例

## 1.1 本章目标

本章先把整套项目的学习路线放在一张地图里。你会看到个人量化交易从数据到信号的完整闭环，而不是一开始就陷入某个函数或库的细节。

学完本章后，你应该能：

- 说清楚量化策略从数据到交易信号的大致步骤。
- 理解为什么后续章节会围绕 `data/sample/` 和 `lib/` 展开。
- 看懂第 13 章经典策略与主线多因子案例之间的关系。

如果你同时阅读系统书籍，可以把它当作概念地图；在这个开源项目里，读者主入口始终是 `notebooks/`。

## 1.2 前置条件

- 能打开并运行 Jupyter notebook。
- 对 `DataFrame`、表格行列和简单收益率有基本直觉。
- 已准备好 `Python 3.11` 环境；第 02 章会做更完整的环境检查。

## 1.3 本章输入与输出

本章不依赖外部网络。我们先用一个很小的手写例子跑通流程，再预览真实 ETF 样例数据和 `lib` 的作用。后续章节会把这些步骤逐步展开，并把结果写入 `outputs/results/`。


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
candidates = []
if (start / "pyquant-roadmap" / "lib").exists():
    candidates.append(start / "pyquant-roadmap")
candidates.extend([start, *start.parents])

PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / "lib").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    raise RuntimeError("无法定位 pyquant-roadmap 项目根目录，请从项目根目录或 notebooks 目录运行。")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"


def rel_path(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.name


pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("PROJECT_ROOT = .")
print(f"SAMPLE_DIR = {rel_path(SAMPLE_DIR)}")
print(f"RESULTS_DIR = {rel_path(RESULTS_DIR)}")


PROJECT_ROOT = .
SAMPLE_DIR = data/sample
RESULTS_DIR = outputs/results


## 1.4 全流程地图

量化不是单个模型，也不是单次回测。一个完整闭环通常包括数据、特征、组合、回测、评估和信号输出。先看清这张地图，后面的每章才知道自己在解决哪一段问题。


In [2]:
workflow = pd.DataFrame(
    [
        ("1 数据获取", "拿到真实可用行情", "AKShare ETF 日线接口", "raw OHLCV 表", "04"),
        ("2 字段标准化", "统一后续处理口径", "原始行情", "date/code/open/high/low/close/volume/amount", "04"),
        ("3 清洗与对齐", "让多资产时间序列可比较", "标准 OHLCV", "价格矩阵、收益率矩阵", "05"),
        ("4 因子构建", "把行情转成可计算特征", "价格与成交量", "因子面板", "06"),
        ("5 因子检验", "判断信号是否值得进入组合", "因子与未来收益", "IC、分层收益", "07"),
        ("6 多因子打分", "合成综合排序依据", "多个标准化因子", "综合 score", "07-08"),
        ("7 TopN 组合", "把分数转成持仓", "score 和调仓日", "目标权重", "08"),
        ("8 回测", "把权重变成收益曲线", "价格和目标权重", "策略收益、NAV", "09"),
        ("9 绩效评估", "解释回测结果", "策略收益", "指标、图表、HTML 报告", "10"),
        ("10 最新权重", "给出当前目标持仓", "目标权重矩阵", "target_weights.csv", "11"),
        ("11 订单建议/信号", "把目标权重转成操作清单", "目标权重、价格、当前持仓", "trade_orders.csv", "11"),
    ],
    columns=["步骤", "要解决的问题", "输入", "输出", "对应章节"],
)
display(workflow)


,步骤,要解决的问题,输入,输出,对应章节
0,1 数据获取,拿到真实可用行情,AKShare ETF 日线接口,raw OHLCV 表,04
1,2 字段标准化,统一后续处理口径,原始行情,date/code/open/high/low/close/volume/amount,04
2,3 清洗与对齐,让多资产时间序列可比较,标准 OHLCV,价格矩阵、收益率矩阵,05
3,4 因子构建,把行情转成可计算特征,价格与成交量,因子面板,06
4,5 因子检验,判断信号是否值得进入组合,因子与未来收益,IC、分层收益,07
5,6 多因子打分,合成综合排序依据,多个标准化因子,综合 score,07-08
6,7 TopN 组合,把分数转成持仓,score 和调仓日,目标权重,08
7,8 回测,把权重变成收益曲线,价格和目标权重,策略收益、NAV,09
8,9 绩效评估,解释回测结果,策略收益,指标、图表、HTML 报告,10
9,10 最新权重,给出当前目标持仓,目标权重矩阵,target_weights.csv,11


## 1.5 主线案例：低频 ETF 多因子 TopN

这套 notebook 的主线案例是一个低频 ETF 多因子策略。它的目标不是预测每天涨跌，而是把多个可计算特征合成一个分数，定期选择分数较高的 ETF，形成目标权重。

默认 ETF 池包括：

- `510300` 沪深300ETF
- `510500` 中证500ETF
- `159915` 创业板ETF
- `512100` 中证1000ETF

第 04 章会展示如何用 AKShare 获取真实 ETF 数据，并把数据缓存为后续章节可复用的样例数据。


In [3]:
planned_outputs = pd.DataFrame(
    [
        ("factor_scores.csv", "每只 ETF 每个日期的因子分数", "10-11（06-07讲原理）"),
        ("target_weights.csv", "策略最新目标权重", "08-11"),
        ("strategy_returns.csv", "策略收益、成本、换手和 NAV", "09-10"),
        ("performance_metrics.csv / .md", "年化收益、波动、夏普、回撤等指标", "10"),
        ("nav_curve.png / drawdown_curve.png", "净值曲线和回撤图", "10"),
        ("quantstats_report.html", "可浏览的策略报告", "10-11"),
        ("trade_orders.csv", "基于目标权重的订单建议", "11"),
    ],
    columns=["产物", "含义", "首次出现章节"],
)
planned_outputs["路径"] = planned_outputs["产物"].map(
    lambda name: rel_path(RESULTS_DIR / name.split(" / ")[0])
)
display(planned_outputs)


,产物,含义,首次出现章节,路径
0,factor_scores.csv,每只 ETF 每个日期的因子分数,10-11（06-07讲原理）,outputs/results/factor_scores.csv
1,target_weights.csv,策略最新目标权重,08-11,outputs/results/target_weights.csv
2,strategy_returns.csv,策略收益、成本、换手和 NAV,09-10,outputs/results/strategy_returns.csv
3,performance_metrics.csv / .md,年化收益、波动、夏普、回撤等指标,10,outputs/results/performance_metrics.csv
4,nav_curve.png / drawdown_curve.png,净值曲线和回撤图,10,outputs/results/nav_curve.png
5,quantstats_report.html,可浏览的策略报告,10-11,outputs/results/quantstats_report.html
6,trade_orders.csv,基于目标权重的订单建议,11,outputs/results/trade_orders.csv


## 1.6 手写一个最小闭环

下面用 3 只虚拟 ETF、5 个调仓月末做一个极小例子。这个例子故意不用 `lib`，目的是看清楚核心链条：

`原始价格 -> 标准化 -> 收益率 -> 因子 -> 打分 -> TopN 权重 -> 回测 -> 指标 -> 最新权重 -> 订单建议`

真实项目只是把每一步的数据规模、质量检查和可复用程度提高。


In [4]:
dates = pd.to_datetime(["2024-01-31", "2024-02-29", "2024-03-29", "2024-04-30", "2024-05-31"])
close_values = {
    "ETF_A": [100.0, 103.0, 106.0, 104.0, 109.0],
    "ETF_B": [100.0, 98.0, 101.0, 105.0, 103.0],
    "ETF_C": [100.0, 101.0, 100.0, 99.0, 102.0],
}

raw_prices = pd.DataFrame(
    [
        {"trade_date": dt.strftime("%Y-%m-%d"), "symbol": code, "close_price": close}
        for code, closes in close_values.items()
        for dt, close in zip(dates, closes)
    ]
).sample(frac=1.0, random_state=7).reset_index(drop=True)

display(raw_prices.head(8))

,trade_date,symbol,close_price
0,2024-04-30,ETF_B,105.0000
1,2024-01-31,ETF_B,100.0000
2,2024-01-31,ETF_C,100.0000
3,2024-03-29,ETF_A,106.0000
4,2024-03-29,ETF_C,100.0000
5,2024-01-31,ETF_A,100.0000
6,2024-02-29,ETF_A,103.0000
7,2024-05-31,ETF_C,102.0000


In [5]:
def standardize_tiny_quotes(raw: pd.DataFrame) -> pd.DataFrame:
    out = raw.rename(
        columns={
            "trade_date": "date",
            "symbol": "code",
            "close_price": "close",
        }
    ).copy()
    out["date"] = pd.to_datetime(out["date"])
    out["code"] = out["code"].astype(str)
    out["close"] = pd.to_numeric(out["close"], errors="coerce")
    out = out.dropna(subset=["date", "code", "close"])
    out = out.drop_duplicates(["date", "code"]).sort_values(["date", "code"])
    return out[["date", "code", "close"]].reset_index(drop=True)

# 这一步对应后续数据章节里的字段标准化和 parquet 缓存。
tiny_cache = standardize_tiny_quotes(raw_prices)
price_matrix = tiny_cache.pivot(index="date", columns="code", values="close").sort_index()
returns = price_matrix.pct_change().fillna(0.0)

display(tiny_cache.head(9))
display(price_matrix)
display(returns.round(4))


,date,code,close
0,2024-01-31,ETF_A,100.0000
1,2024-01-31,ETF_B,100.0000
2,2024-01-31,ETF_C,100.0000
3,2024-02-29,ETF_A,103.0000
4,2024-02-29,ETF_B,98.0000
5,2024-02-29,ETF_C,101.0000
6,2024-03-29,ETF_A,106.0000
7,2024-03-29,ETF_B,101.0000
8,2024-03-29,ETF_C,100.0000


code,ETF_A,ETF_B,ETF_C
date,,,
2024-01-31,100.0000,100.0000,100.0000
2024-02-29,103.0000,98.0000,101.0000
2024-03-29,106.0000,101.0000,100.0000
2024-04-30,104.0000,105.0000,99.0000
2024-05-31,109.0000,103.0000,102.0000


code,ETF_A,ETF_B,ETF_C
date,,,
2024-01-31,0.0000,0.0000,0.0000
2024-02-29,0.0300,-0.0200,0.0100
2024-03-29,0.0291,0.0306,-0.0099
2024-04-30,-0.0189,0.0396,-0.0100
2024-05-31,0.0481,-0.0190,0.0303


### 1.6.1 构造两个最小因子

这里用两个非常简单的因子：

- `momentum_1`：最近一期收益率，代表短期动量。
- `defensive_2`：最近两期波动率的相反数，代表低波动偏好。

重点不是因子本身，而是看清楚“先按资产计算时序特征，再按日期做横截面比较”的流程。


In [6]:
def zscore_series(s: pd.Series) -> pd.Series:
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / std

momentum_1 = returns
# rolling 需要足够历史窗口，前几期自然会出现 NaN。
defensive_2 = -returns.rolling(2).std(ddof=0)

factors = (
    momentum_1.stack().rename("momentum_1").to_frame()
    .join(defensive_2.stack().rename("defensive_2"))
    .reset_index()
    .rename(columns={"level_0": "date", "level_1": "code"})
)

factor_cols = ["momentum_1", "defensive_2"]
factors = factors.dropna(subset=factor_cols).copy()
for col in factor_cols:
    factors[f"{col}_z"] = factors.groupby("date")[col].transform(zscore_series)

factors["score"] = 0.70 * factors["momentum_1_z"] + 0.30 * factors["defensive_2_z"]

display(factors.round(4))


,date,code,momentum_1,defensive_2,momentum_1_z,defensive_2_z,score
3,2024-02-29,ETF_A,0.0300,-0.0150,1.1355,-1.2247,0.4275
4,2024-02-29,ETF_B,-0.0200,-0.0100,-1.2978,0.0000,-0.9084
5,2024-02-29,ETF_C,0.0100,-0.0050,0.1622,1.2247,0.4810
6,2024-03-29,ETF_A,0.0291,-0.0004,0.6671,1.1186,0.8026
7,2024-03-29,ETF_B,0.0306,-0.0253,0.7463,-1.3087,0.1298
8,2024-03-29,ETF_C,-0.0099,-0.0100,-1.4135,0.1901,-0.9324
9,2024-04-30,ETF_A,-0.0189,-0.0240,-0.8724,-1.3925,-1.0284
10,2024-04-30,ETF_B,0.0396,-0.0045,1.4001,0.4825,1.1249
11,2024-04-30,ETF_C,-0.0100,-0.0000,-0.5277,0.9100,-0.0964
12,2024-05-31,ETF_A,0.0481,-0.0335,0.9966,-1.0461,0.3838


### 1.6.2 做一个最小因子体检

因子算出来不等于有效。这里用下一期收益做一个极简 rank IC 检查，让你先建立“信号需要被验证”的意识。真实检验会在第 07 章展开。


In [7]:
future_return_panel = (
    returns.shift(-1).stack().rename("next_return").reset_index()
    .rename(columns={"level_0": "date", "level_1": "code"})
)
check_panel = factors.merge(future_return_panel, on=["date", "code"], how="left", validate="one_to_one")

def daily_rank_ic(panel: pd.DataFrame, factor_col: str) -> pd.DataFrame:
    rows = []
    clean = panel.dropna(subset=[factor_col, "next_return"])
    for dt, group in clean.groupby("date"):
        if len(group) < 2:
            continue
        rank_ic = group[factor_col].rank().corr(group["next_return"].rank())
        rows.append({"date": dt, "factor": factor_col, "rank_ic": rank_ic})
    return pd.DataFrame(rows)

ic_table = pd.concat(
    [daily_rank_ic(check_panel, col) for col in ["momentum_1", "defensive_2", "score"]],
    ignore_index=True,
)

display(check_panel[["date", "code", "momentum_1", "defensive_2", "score", "next_return"]].round(4))
display(ic_table.round(4))

,date,code,momentum_1,defensive_2,score,next_return
0,2024-02-29,ETF_A,0.0300,-0.0150,0.4275,0.0291
1,2024-02-29,ETF_B,-0.0200,-0.0100,-0.9084,0.0306
2,2024-02-29,ETF_C,0.0100,-0.0050,0.4810,-0.0099
3,2024-03-29,ETF_A,0.0291,-0.0004,0.8026,-0.0189
4,2024-03-29,ETF_B,0.0306,-0.0253,0.1298,0.0396
5,2024-03-29,ETF_C,-0.0099,-0.0100,-0.9324,-0.0100
6,2024-04-30,ETF_A,-0.0189,-0.0240,-1.0284,0.0481
7,2024-04-30,ETF_B,0.0396,-0.0045,1.1249,-0.0190
8,2024-04-30,ETF_C,-0.0100,-0.0000,-0.0964,0.0303
9,2024-05-31,ETF_A,0.0481,-0.0335,0.3838,NaN


,date,factor,rank_ic
0,2024-02-29,momentum_1,-0.5000
1,2024-03-29,momentum_1,0.5000
2,2024-04-30,momentum_1,-1.0000
3,2024-02-29,defensive_2,-0.5000
4,2024-03-29,defensive_2,-1.0000
5,2024-04-30,defensive_2,-0.5000
6,2024-02-29,score,-1.0000
7,2024-03-29,score,-0.5000
8,2024-04-30,score,-1.0000


### 1.6.3 从分数到权重、回测和订单建议

有了 `score` 之后，下一步不是直接交易，而是先把分数转成目标权重。这里选择 Top2 等权，再用滞后一日持仓做一个透明的向量化回测。


In [8]:
def choose_top_n_equal_weight(scores: pd.DataFrame, n: int = 2) -> pd.DataFrame:
    rows = []
    for dt, group in scores.dropna(subset=["score"]).groupby("date"):
        top = group.sort_values("score", ascending=False).head(n)
        if top.empty:
            continue
        rows.append(pd.DataFrame({"date": dt, "code": top["code"].to_numpy(), "weight": 1.0 / len(top)}))
    if not rows:
        return pd.DataFrame(columns=["date", "code", "weight"])
    return pd.concat(rows, ignore_index=True).sort_values(["date", "code"]).reset_index(drop=True)

def weight_matrix_from_long(weights: pd.DataFrame, index: pd.Index, columns: pd.Index) -> pd.DataFrame:
    matrix = weights.pivot(index="date", columns="code", values="weight")
    return matrix.reindex(index=index, columns=columns).ffill().fillna(0.0)

def vector_backtest(price: pd.DataFrame, target_weights: pd.DataFrame, cost_bps: float = 5.0) -> pd.DataFrame:
    asset_returns = price.pct_change().fillna(0.0)
    weights = target_weights.reindex(index=price.index, columns=price.columns).fillna(0.0)
    positions = weights.shift(1).fillna(0.0)
    gross_return = (positions * asset_returns).sum(axis=1)
    turnover = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    cost = turnover * cost_bps / 10_000
    strategy_return = gross_return - cost
    nav = (1.0 + strategy_return).cumprod()
    return pd.DataFrame(
        {
            "gross_return": gross_return,
            "cost": cost,
            "strategy_return": strategy_return,
            "nav": nav,
            "turnover": turnover,
        }
    )

tiny_weights = choose_top_n_equal_weight(factors, n=2)
tiny_weight_matrix = weight_matrix_from_long(tiny_weights, price_matrix.index, price_matrix.columns)
tiny_report = vector_backtest(price_matrix, tiny_weight_matrix, cost_bps=5.0)

display(tiny_weights)
display(tiny_weight_matrix.round(2))
display(tiny_report.round(4))

,date,code,weight
0,2024-02-29,ETF_A,0.5000
1,2024-02-29,ETF_C,0.5000
2,2024-03-29,ETF_A,0.5000
3,2024-03-29,ETF_B,0.5000
4,2024-04-30,ETF_B,0.5000
5,2024-04-30,ETF_C,0.5000
6,2024-05-31,ETF_A,0.5000
7,2024-05-31,ETF_C,0.5000


code,ETF_A,ETF_B,ETF_C
date,,,
2024-01-31,0.0000,0.0000,0.0000
2024-02-29,0.5000,0.0000,0.5000
2024-03-29,0.5000,0.5000,0.5000
2024-04-30,0.5000,0.5000,0.5000
2024-05-31,0.5000,0.5000,0.5000


,gross_return,cost,strategy_return,nav,turnover
date,,,,,
2024-01-31,0.0000,0.0000,0.0000,1.0000,0.0000
2024-02-29,0.0000,0.0005,-0.0005,0.9995,1.0000
2024-03-29,0.0096,0.0002,0.0094,1.0089,0.5000
2024-04-30,0.0054,0.0000,0.0054,1.0143,0.0000
2024-05-31,0.0297,0.0000,0.0297,1.0444,0.0000


In [9]:
def summarize_backtest(report: pd.DataFrame) -> pd.DataFrame:
    nav = report["nav"]
    drawdown = nav / nav.cummax() - 1.0
    return pd.DataFrame(
        [
            {
                "total_return": nav.iloc[-1] - 1.0,
                "max_drawdown": drawdown.min(),
                "avg_turnover": report["turnover"].mean(),
                "last_nav": nav.iloc[-1],
            }
        ]
    )

def latest_weights_from_matrix(weights: pd.DataFrame) -> pd.DataFrame:
    latest_date = weights.index.max()
    latest = weights.loc[latest_date]
    out = latest[latest > 0].rename("target_weight").reset_index()
    out.columns = ["code", "target_weight"]
    out.insert(0, "date", latest_date)
    return out.sort_values("code").reset_index(drop=True)

def target_orders(latest_weights: pd.DataFrame, latest_prices: pd.Series, capital: float, lot_size: int = 100) -> pd.DataFrame:
    rows = []
    for _, row in latest_weights.iterrows():
        code = row["code"]
        target_weight = float(row["target_weight"])
        price = float(latest_prices.loc[code])
        target_value = capital * target_weight
        target_shares = int(np.floor(target_value / price / lot_size) * lot_size)
        rows.append(
            {
                "signal_date": row["date"],
                "code": code,
                "side": "BUY" if target_shares > 0 else "HOLD",
                "price": price,
                "target_weight": target_weight,
                "target_shares": target_shares,
                "order_value": target_shares * price,
            }
        )
    return pd.DataFrame(rows)

latest_tiny_weights = latest_weights_from_matrix(tiny_weight_matrix)
latest_tiny_orders = target_orders(
    latest_tiny_weights,
    latest_prices=price_matrix.loc[price_matrix.index.max()],
    capital=100_000,
    lot_size=100,
)

display(summarize_backtest(tiny_report).round(4))
display(latest_tiny_weights)
display(latest_tiny_orders)

,total_return,max_drawdown,avg_turnover,last_nav
0,0.0444,-0.0005,0.3000,1.0444


,date,code,target_weight
0,2024-05-31,ETF_A,0.5000
1,2024-05-31,ETF_B,0.5000
2,2024-05-31,ETF_C,0.5000


,signal_date,code,side,price,target_weight,target_shares,order_value
0,2024-05-31,ETF_A,BUY,109.0000,0.5000,400,"43,600.0000"
1,2024-05-31,ETF_B,BUY,103.0000,0.5000,400,"41,200.0000"
2,2024-05-31,ETF_C,BUY,102.0000,0.5000,400,"40,800.0000"


### 1.6.4 小练习：改变 TopN

把 TopN 从 2 改成 1，观察集中持仓后的净值、回撤和换手变化。这个练习帮助你理解“选几个资产”本身就是策略规则的一部分。


In [10]:
top1_weights = choose_top_n_equal_weight(factors, n=1)
top1_matrix = weight_matrix_from_long(top1_weights, price_matrix.index, price_matrix.columns)
top1_report = vector_backtest(price_matrix, top1_matrix, cost_bps=5.0)

exercise_summary = pd.concat(
    [
        summarize_backtest(tiny_report).assign(case="Top2 equal weight"),
        summarize_backtest(top1_report).assign(case="Top1 concentrated"),
    ],
    ignore_index=True,
)[["case", "total_return", "max_drawdown", "avg_turnover", "last_nav"]]

display(top1_weights.tail(5))
display(exercise_summary.round(4))

,date,code,weight
0,2024-02-29,ETF_C,1.0000
1,2024-03-29,ETF_A,1.0000
2,2024-04-30,ETF_B,1.0000
3,2024-05-31,ETF_C,1.0000


,case,total_return,max_drawdown,avg_turnover,last_nav
0,Top2 equal weight,0.0444,-0.0005,0.3000,1.0444
1,Top1 concentrated,0.0170,-0.0399,0.6000,1.0170


## 1.7 从手写原理过渡到 `lib`

后续章节会先手写最小函数讲清楚原理，再使用成熟库和 `lib` 复现实战做法：

- `pandas` / `numpy` 负责表格、矩阵和向量化计算。
- `ta`、`bt`、`quantstats` 分别负责常见技术指标、权重型回测和报告能力。
- `lib/` 把本项目中反复使用的项目逻辑沉淀下来，例如字段标准化、因子面板、组合权重、回测收益、绩效报告和订单建议。

这样做既避免过度手搓，也不会在读者理解原理之前把关键过程藏进黑盒。


In [11]:
from lib.data import load_sample_assets, load_sample_prices

assets = load_sample_assets()
prices = load_sample_prices()

price_summary = (
    prices.groupby("code")
    .agg(start=("date", "min"), end=("date", "max"), rows=("date", "size"), first_close=("close", "first"), last_close=("close", "last"))
    .reset_index()
    .merge(assets[["code", "name", "asset_type"]], on="code", how="left")
    [["code", "name", "asset_type", "start", "end", "rows", "first_close", "last_close"]]
)

display(assets)
display(price_summary)

,code,name,asset_type,list_date
0,510300,沪深300ETF,ETF,2000-01-01
1,510500,中证500ETF,ETF,2000-01-01
2,159915,创业板ETF,ETF,2000-01-01
3,512100,中证1000ETF,ETF,2000-01-01


,code,name,asset_type,start,end,rows,first_close,last_close
0,159915,创业板ETF,ETF,2021-01-04,2023-12-29,725,2.9760,1.8420
1,510300,沪深300ETF,ETF,2021-01-04,2023-12-29,725,4.8430,3.2190
2,510500,中证500ETF,ETF,2021-01-04,2023-12-29,725,6.0140,5.2790
3,512100,中证1000ETF,ETF,2021-01-04,2023-12-29,725,2.5370,2.2960


In [12]:
from lib.backtest import returns_from_weights
from lib.evaluation.metrics import perf_stats
from lib.factors import build_technical_factor_panel, combine_score, zscore_by_date
from lib.portfolio import top_n_equal_weight, weights_to_matrix
from lib.trading import latest_target_weights, target_weights_to_orders

sample_codes = assets.loc[assets["asset_type"].eq("ETF"), "code"].astype(str).tolist()
sample_prices = prices[prices["code"].isin(sample_codes)].copy()
close = sample_prices.pivot(index="date", columns="code", values="close").sort_index().ffill().dropna()

factor_columns = ["momentum_60", "low_vol_20", "ma_gap_20_60"]
factor_weights = {"momentum_60": 0.45, "low_vol_20": 0.35, "ma_gap_20_60": 0.20}
factor_panel = build_technical_factor_panel(sample_prices, sample_codes)
scored = combine_score(zscore_by_date(factor_panel, factor_columns), factor_weights)

decision_dates = pd.Series(pd.to_datetime(scored["date"].drop_duplicates())).sort_values()
month_end_decisions = decision_dates.groupby(decision_dates.dt.to_period("M")).max()
rebalance_scores = scored[scored["date"].isin(month_end_decisions)]
sparse_weights = top_n_equal_weight(rebalance_scores, n=3)
target_weight_matrix = weights_to_matrix(sparse_weights, close.index, close.columns, carry_forward=True)
preview_report = returns_from_weights(close, target_weight_matrix, cost_bps=8.0)

latest_scores = (
    scored[scored["date"].eq(scored["date"].max())]
    .sort_values("score", ascending=False)
    .merge(assets[["code", "name"]], on="code", how="left")
    [["date", "code", "name", *factor_columns, "score"]]
)
latest_weights = latest_target_weights(target_weight_matrix).merge(assets[["code", "name"]], on="code", how="left")
orders = target_weights_to_orders(target_weight_matrix, close, capital=1_000_000, lot_size=100)
preview_stats = pd.DataFrame([perf_stats(preview_report["strategy_return"])]).T.reset_index()
preview_stats.columns = ["metric", "value"]

preview_objects = pd.DataFrame(
    [
        ("factor_panel", len(factor_panel), "技术因子面板"),
        ("scored", len(scored), "多因子综合分数"),
        ("sparse_weights", len(sparse_weights), "月末 TopN 调仓权重"),
        ("preview_report", len(preview_report), "透明回测结果"),
    ],
    columns=["对象", "行数", "说明"],
)

display(preview_objects)
display(latest_scores.round(4))
display(latest_weights[["date", "code", "name", "target_weight"]].round(4))
display(orders.head(10).round(4))
display(preview_stats.round(4))


,对象,行数,说明
0,factor_panel,2656,技术因子面板
1,scored,2656,多因子综合分数
2,sparse_weights,99,月末 TopN 调仓权重
3,preview_report,725,透明回测结果


,date,code,name,momentum_60,low_vol_20,ma_gap_20_60,score
0,2023-12-29,512100,中证1000ETF,-0.0387,-0.0104,-0.0130,0.8990
1,2023-12-29,510500,中证500ETF,-0.0557,-0.0085,-0.0165,0.6667
2,2023-12-29,510300,沪深300ETF,-0.0826,-0.0100,-0.0417,-0.7652
3,2023-12-29,159915,创业板ETF,-0.0629,-0.0141,-0.0357,-0.8006


,date,code,name,target_weight
0,2023-12-29,510300,沪深300ETF,0.3333
1,2023-12-29,510500,中证500ETF,0.3333
2,2023-12-29,512100,中证1000ETF,0.3333


,signal_date,code,side,price,target_weight,current_weight,delta_weight,target_shares,current_shares,order_shares,order_value
0,2023-12-29,510300,BUY,3.2190,0.3333,0.0000,0.3333,103500,0,103500,"333,166.5000"
1,2023-12-29,510500,BUY,5.2790,0.3333,0.0000,0.3333,63100,0,63100,"333,104.9000"
2,2023-12-29,512100,BUY,2.2960,0.3333,0.0000,0.3333,145100,0,145100,"333,149.6000"


,metric,value
0,ann_return,-0.0898
1,ann_vol,0.1731
2,sharpe,-0.4567
3,max_drawdown,-0.3661
4,total_return,-0.2372
5,win_rate,0.4359


## 1.8 本章小结与下一章衔接

本章建立的是路线图：

- 数据先标准化和缓存，后续步骤才能稳定复用。
- 因子是从行情数据中计算出来的特征。
- 分数要先转成权重，再进入回测。
- 回测结果需要指标、图表和报告解释。
- 最新目标权重还要转成订单建议，才算接近交易信号。

第 02 章会检查环境、项目结构和第一次运行，确保你后面可以按顺序执行所有 notebook。
